In [4]:
from input_pipeline import preprocess_dataset, get_masked_dataset, batch_sampler
import xarray as xr
import jax.numpy as jnp
import jax
import chex
import numpy as np
import flax
import rich
import flax.linen as nn
import optax

### Load, Normalization & Masking

In [5]:
ds = xr.load_dataarray('../data/SiC_100x10.nc')
filtered_ds = preprocess_dataset(ds, verbose = True)
masked_ds = get_masked_dataset(filtered_ds)

Dropped 20 spectra


In [6]:
# Transformer model, following "Formal algorithms for transformers" [arXiv:2207.09238 [cs.LG]] 

class LinearProjection(nn.Module):
    """Embedding layer for transformer model"""
    embedding_dim: int

    @nn.compact
    def __call__(self, x):
        x = nn.Dense(self.embedding_dim, use_bias=False)(x)
        x = nn.LayerNorm()(x)
        return x
    
class FFBlock(nn.Module):
    """Feed-forward block for transformer model"""
    embedding_dim: int
    dropout_rate: float = 0.1

    @nn.compact
    def __call__(self, x, training: bool):
        x = nn.Dense(4*self.embedding_dim)(x)
        x = nn.relu(x)
        x = nn.Dense(self.embedding_dim)(x)
        x = nn.Dropout(self.dropout_rate, deterministic = not training)(x)
        return x

class TransformerEncoderLayer(nn.Module):
    """Transformer encoder layer"""
    embedding_dim: int
    num_heads: int
    dropout_rate: float = 0.1

    @nn.compact
    def __call__(self, x):
        # Multi-head attention
        x = nn.LayerNorm()(x)
        x = nn.MultiHeadDotProductAttention(
            num_heads = self.num_heads,
            qkv_features = self.embedding_dim,
            dropout_rate = self.dropout_rate
        )(x)
        x = x + nn.Dense(self.embedding_dim)(x)
        x = nn.LayerNorm()(x)
        x = FFBlock(self.embedding_dim, self.dropout_rate)(x)
        x = x + nn.Dense(self.embedding_dim)(x)
        return x

In [7]:
counts_emb = LinearProjection(64)
wn_emb = LinearProjection(64)
train_iter = batch_sampler(filtered_ds, masked_ds, batch_size = None, shuffle = True, drop_last = True)
batch = next(train_iter)

2023-12-06 22:04:43.014085: W external/xla/xla/service/gpu/nvptx_compiler.cc:698] The NVIDIA driver's CUDA version is 12.1 which is older than the ptxas CUDA version (12.3.103). Because the driver is older than the ptxas version, XLA is disabling parallel compilation, which may slow down compilation. You should update your NVIDIA driver or use the NVIDIA-provided CUDA forward compatibility packages.


In [8]:
counts_vars = counts_emb.init(jax.random.PRNGKey(0), batch['spectra'][0])
wn_vars = wn_emb.init(jax.random.PRNGKey(0), batch['wave_number'][0])

In [9]:
embc = counts_emb.apply(counts_vars, batch['spectra'])
embwn = wn_emb.apply(wn_vars, batch['wave_number'])

In [10]:
x = embc + embwn

In [11]:
x.shape

(980, 1015, 64)

In [12]:
ff = FFBlock(64)
ff_vars = ff.init(jax.random.PRNGKey(0), x[0], training = False)
x = ff.apply(ff_vars, x, training = True, rngs={'dropout': jax.random.PRNGKey(0)})

In [13]:
tabulate_fn = nn.tabulate(ff, jax.random.PRNGKey(0), compute_flops=True)

In [90]:
x.shape

(980, 1015, 64)